In [1]:
# Check if using Google Colab
try:
    from google.colab import drive

    USE_COLAB = True

    print("Running in Google Colab environment.")
except ImportError:
    USE_COLAB = False
    print("Running in local environment.")

Running in local environment.


In [ ]:
## Imports

import numpy as np
import os
import pandas as pd

import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    StratifiedKFold,
    LeaveOneGroupOut,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from xgboost import XGBClassifier

import torch
from torch.utils.data import Dataset

from dotenv import load_dotenv

In [ ]:
if USE_COLAB:
    # Access Google Drive files

    from google.colab import drive
    import os

    # get data
    GOOGLE_ROOT = "/content/drive/MyDrive/research_data/mind-wandering/Brandmeyer/"
    drive.mount("/content/drive")
    os.chdir(GOOGLE_ROOT)

    print(f"Current Directory: {os.getcwd()}")

In [9]:
## Settings and Info

# Paths
load_dotenv()  # Load environment variables from .env file
ROOT = os.getenv("ROOT")
if USE_COLAB:
    ROOT = GOOGLE_ROOT

RAW_DATA = ROOT
DERIVATIVES = f"{ROOT}derivatives"
DATA_PATH = f"{DERIVATIVES}/noica"
FEATURES_PATH = f"{DATA_PATH}/features"
MODEL_RESULTS_PATH = f"{DATA_PATH}/model_results"

N_JOBS = -1  # Num of cores to use for parallel processing (adjust as needed)

# Frequency bands
BANDS = {
    "delta": [2, 4],
    "theta": [4, 8],
    "alpha": [8, 12],
    "beta": [12, 30],
    "gamma": [30, 50],
}

SUBJECTS = [f"{i:03d}" for i in range(1, 25)]  # '001' through '024'
SESSIONS = ["01", "02", "03"]

In [10]:
class EEGDataset(Dataset):
    def __init__(self, raw_data, exclude_subjects=None):
        ## Filter subjects
        if exclude_subjects is not None:
            raw_data = raw_data[~raw_data["subject"].isin(exclude_subjects)].copy()

        ## Clean and label data
        # Filter rows where mw_depth equals meditation_depth
        raw_data = raw_data[raw_data["mw_depth"] != raw_data["meditation_depth"]].copy()
        # Label data: is_mw variable is 1 for mind wandering, 0 for meditation
        raw_data["is_mw"] = (
            raw_data["mw_depth"] > raw_data["meditation_depth"]
        ).astype(int)

        raw_data = self._add_relative_power(raw_data)

        self.raw_data = raw_data
        non_feature_cols = [
            "subject",
            "session",
            "channel",
            "epoch_idx",
            "is_mw",
            "mw_depth",
            "meditation_depth",
            "fatigue",
        ]
        feature_columns = [
            col for col in self.raw_data.columns if col not in non_feature_cols
        ]

        self.features = feature_columns

    def _prepare_xy(self, df):
        """Helper to handle the pivot logic consistently"""
        # Pivot to get (Subject, Session, Epoch, Label) x (Channels * Features)
        widened = df.pivot(
            index=["subject", "session", "epoch_idx", "is_mw"],
            columns="channel",
            values=self.features,
        ).reset_index()

        X = widened.drop(columns=["subject", "session", "epoch_idx", "is_mw"])
        y = widened["is_mw"]
        groups = widened["subject"].to_numpy()
        return X, y, groups

    def get_data(self):
        return self._prepare_xy(self.raw_data)

    def get_normalized_data(self, groups=["subject", "session"]):
        """
        z-score normalization;
        defaults to normalizing within subject, channel, and session to preserve individual differences and channel characteristics
        """
        data_normalized = self.raw_data.copy()
        data_normalized[self.features] = data_normalized.groupby(groups)[
            self.features
        ].transform(lambda x: (x - x.mean()) / (x.std() + 1e-4))

        return self._prepare_xy(data_normalized)

    def get_shuffled_data(self, groups=["subject"], random_state=42):
        """
        Shuffles labels (is_mw) within specified groups to preserve
        subject-specific class imbalances while destroying the
        feature-to-label relationship.
        """

        # Set the seed for reproducibility
        np.random.seed(random_state)

        # Pivot to wide format
        widened = self.raw_data.pivot(
            index=["subject", "session", "epoch_idx", "is_mw"],
            columns="channel",
            values=self.features,
        ).reset_index()

        # Perform group-wise permutation
        widened["is_mw"] = widened.groupby(groups)["is_mw"].transform(
            lambda x: np.random.permutation(x.values)
        )

        X = widened.drop(columns=["subject", "session", "epoch_idx", "is_mw"])
        y = widened["is_mw"]
        subj_groups = widened["subject"].to_numpy()

        return X, y, subj_groups

    def _add_relative_power(self, df):
        """
        Calculates relative power for specified bands and appends them to the DataFrame.
        """

        bands = [
            "delta_mean_power",
            "theta_mean_power",
            "alpha_mean_power",
            "beta_mean_power",
            "gamma_mean_power",
        ]

        # Calculate total power across the specified frequency bands
        # We replace 0 with a tiny value to prevent division by zero errors
        total_power = df[bands].sum(axis=1).replace(0, 1e-10)

        for band in bands:
            df[f"rel_{band}"] = df[band] / total_power

        return df

In [11]:
class ColumnSelector(BaseEstimator, TransformerMixin):
    # Allows selecting features by column name in a sklearn pipeline
    def __init__(self, cols=None):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # If no columns specified, return everything
        if self.cols is None:
            return X
        return X[self.cols]

In [12]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split


class LeavePartialGroupOut:
    def __init__(self, n_groups=4, p_split=0.5, n_iterations=10, random_state=42):
        """
        n_groups: The 2n subjects to pick for the test pool.
        p_split: Percentage of those subjects' epochs to withhold for testing.
        n_iterations: Total number of iterations to run (prevents the 10k+ explosion).
        """
        self.gss = GroupShuffleSplit(
            n_splits=n_iterations, test_size=n_groups, random_state=random_state
        )
        self.p_split = p_split
        self.random_state = random_state

    def split(self, X, y, groups):
        for other_subs_idx, test_pool_idx in self.gss.split(X, y, groups):
            final_train_idx = list(other_subs_idx)
            final_test_idx = []

            # Identify the unique subjects in the test pool
            test_subjects = np.unique(groups[test_pool_idx])

            for sub in test_subjects:
                # Get indices for JUST this specific subject
                sub_indices = np.where(groups == sub)[0]

                # Get labels for this subject to check for sparsity
                y_sub = y.iloc[sub_indices] if hasattr(y, "iloc") else y[sub_indices]
                unique_classes, counts = np.unique(y_sub, return_counts=True)

                # check if we can stratify
                # Need at least 2 samples of every class present
                can_stratify = len(unique_classes) > 1 and np.all(counts >= 2)

                # Split this subject's epochs individually
                sub_train, sub_test = train_test_split(
                    sub_indices,
                    test_size=self.p_split,
                    random_state=self.random_state,
                    stratify=(y_sub if can_stratify else None),
                )

                final_train_idx.extend(sub_train)
                final_test_idx.extend(sub_test)

            yield np.array(final_train_idx), np.array(final_test_idx)

    def get_n_splits(self, X, y, groups):
        return self.gss.n_splits

In [13]:
# Load raw data
raw_data = pd.read_csv(
    f"{FEATURES_PATH}/combined_features/all_features_merged.tsv", sep="\t"
)
subjs_to_exclude = ["009", "017", "018"]  # Exclude subjects with sparse labels
dataset = EEGDataset(raw_data=raw_data, exclude_subjects=subjs_to_exclude)

# define all features
all_features = [
    "delta_mean_power",
    "theta_mean_power",
    "alpha_mean_power",
    "beta_mean_power",
    "gamma_mean_power",
    "rel_delta_mean_power",
    "rel_theta_mean_power",
    "rel_alpha_mean_power",
    "rel_beta_mean_power",
    "rel_gamma_mean_power",
    "delta_flat_mean_power",
    "theta_flat_mean_power",
    "alpha_flat_mean_power",
    "beta_flat_mean_power",
    "gamma_flat_mean_power",
    "aperiodic_offset",
    "aperiodic_exponent",
    "r_squared",
    "error",
]

In [14]:
# Compare relevant features
def compare_features(
    dataset,
    classifier,
    features,
    cv,
    normalize_data=False,
    shuffle_labels=False,
):
    scoring = ["accuracy", "f1_weighted", "f1_macro"]

    if normalize_data:
        X, y, groups = dataset.get_normalized_data()
    elif shuffle_labels:
        X, y, groups = dataset.get_shuffled_data()
    else:
        X, y, groups = dataset.get_data()

    all_results = []
    for feature_name, feature_list in features.items():

        model = Pipeline(
            [
                ("selector", ColumnSelector(feature_list)),
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=10)),
                classifier,
            ]
        )

        results = cross_validate(
            model,
            X=X,
            y=y,
            cv=cv,
            groups=groups,
            return_train_score=True,
            scoring=scoring,
        )

        row = {"feature": feature_name}
        for metric in scoring:
            row[f"train_{metric}_mean"] = results[f"train_{metric}"].mean()
            # row[f"train_{metric}_std"] = results[f"train_{metric}"].std()
            row[f"test_{metric}_mean"] = results[f"test_{metric}"].mean()
            # row[f"test_{metric}_std"] = results[f"test_{metric}"].std()

        all_results.append(row)

        print(f"Completed cross-validation for feature(s): {feature_name}")

    results_df = pd.DataFrame(all_results)
    return results_df

In [15]:
features = {
    "delta_mean_power": ["delta_mean_power"],
    "theta_mean_power": ["theta_mean_power"],
    "alpha_mean_power": ["alpha_mean_power"],
    "beta_mean_power": ["beta_mean_power"],
    "gamma_mean_power": ["gamma_mean_power"],
    "rel_delta_mean_power": ["rel_delta_mean_power"],
    "rel_theta_mean_power": ["rel_theta_mean_power"],
    "rel_alpha_mean_power": ["rel_alpha_mean_power"],
    "rel_beta_mean_power": ["rel_beta_mean_power"],
    "rel_gamma_mean_power": ["rel_gamma_mean_power"],
    "delta_flat_mean_power": ["delta_flat_mean_power"],
    "theta_flat_mean_power": ["theta_flat_mean_power"],
    "alpha_flat_mean_power": ["alpha_flat_mean_power"],
    "beta_flat_mean_power": ["beta_flat_mean_power"],
    "gamma_flat_mean_power": ["gamma_flat_mean_power"],
    "aperiodic_offset": ["aperiodic_offset"],
    "aperiodic_exponent": ["aperiodic_exponent"],
    "error": ["error"],
    "all_mean_power": [
        "delta_mean_power",
        "theta_mean_power",
        "alpha_mean_power",
        "beta_mean_power",
        "gamma_mean_power",
    ],
    "all_rel_mean_power": [
        "rel_delta_mean_power",
        "rel_theta_mean_power",
        "rel_alpha_mean_power",
        "rel_beta_mean_power",
        "rel_gamma_mean_power",
    ],
    "all_flat_mean_power": [
        "delta_flat_mean_power",
        "theta_flat_mean_power",
        "alpha_flat_mean_power",
        "beta_flat_mean_power",
        "gamma_flat_mean_power",
    ],
    "all_aperiodic": ["aperiodic_offset", "aperiodic_exponent"],
    "all_feats": [
        "delta_mean_power",
        "theta_mean_power",
        "alpha_mean_power",
        "beta_mean_power",
        "gamma_mean_power",
        "rel_delta_mean_power",
        "rel_theta_mean_power",
        "rel_alpha_mean_power",
        "rel_beta_mean_power",
        "rel_gamma_mean_power",
        "delta_flat_mean_power",
        "theta_flat_mean_power",
        "alpha_flat_mean_power",
        "beta_flat_mean_power",
        "gamma_flat_mean_power",
        "aperiodic_offset",
        "aperiodic_exponent",
    ],
}

n_splits = 50
cvs = {
    "StratifiedKFold": StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=42
    ),
    "LeaveOneGroupOut": LeaveOneGroupOut(),
    "Leave2Out": GroupShuffleSplit(
        n_splits=n_splits, test_size=2, random_state=42
    ),  # Leave 2 subjs out
    "LeavePartialGroupOut": LeavePartialGroupOut(
        n_groups=4, p_split=0.5, n_iterations=n_splits, random_state=42
    ),  # Leave 50% of 4 subjs out
}

classifiers = {
    "svm": (
        "svm",
        SVC(kernel="rbf", C=0.1, gamma="scale", class_weight="balanced"),
    ),
    "xgboost": (
        "xgb",
        XGBClassifier(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.05,
            subsample=0.7,
            colsample_bytree=1.0,
            gamma=1.0,
            reg_lambda=10.0,
            reg_alpha=5.0,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
    ),
}


all_results = []
for clf_name, clf in classifiers.items():
    print(f"Classifier: {clf_name}")
    for cv_name, cv in cvs.items():
        print(f"Evaluating classifiers with {cv_name} cross-validation:")
        result_df = compare_features(
            dataset=dataset,
            classifier=clf,
            features=features,
            cv=cv,
            normalize_data=False,
        )
        result_df["classifier_key"] = clf_name
        result_df["classifier_label"] = clf_name
        result_df["cv"] = cv_name
        all_results.append(result_df)

# save results
final_results_df = pd.concat(all_results, ignore_index=True)
final_results_df.to_csv(
    f"{MODEL_RESULTS_PATH}/model_comparison_results.tsv", sep="\t", index=False
)
print("Model comparison complete. Results saved to model_comparison_results.tsv")

Classifier: svm
Evaluating classifiers with StratifiedKFold cross-validation:


/var/folders/mt/wkhh8vn92hxf7nvqmjyjs4qc0000gn/T/ipykernel_54055/3097016513.py:43: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  X = widened.drop(columns=["subject", "session", "epoch_idx", "is_mw"])
/Users/maxboyington/Applications/MNE-Python/1.9.0_0/.mne-python/envs/my-default/lib/python3.13/site-packages/sklearn/model_selection/_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
features = {
    "delta_mean_power": ["delta_mean_power"],
    "theta_mean_power": ["theta_mean_power"],
    "alpha_mean_power": ["alpha_mean_power"],
    "beta_mean_power": ["beta_mean_power"],
    "gamma_mean_power": ["gamma_mean_power"],
    "rel_delta_mean_power": ["rel_delta_mean_power"],
    "rel_theta_mean_power": ["rel_theta_mean_power"],
    "rel_alpha_mean_power": ["rel_alpha_mean_power"],
    "rel_beta_mean_power": ["rel_beta_mean_power"],
    "rel_gamma_mean_power": ["rel_gamma_mean_power"],
    "delta_flat_mean_power": ["delta_flat_mean_power"],
    "theta_flat_mean_power": ["theta_flat_mean_power"],
    "alpha_flat_mean_power": ["alpha_flat_mean_power"],
    "beta_flat_mean_power": ["beta_flat_mean_power"],
    "gamma_flat_mean_power": ["gamma_flat_mean_power"],
    "aperiodic_offset": ["aperiodic_offset"],
    "aperiodic_exponent": ["aperiodic_exponent"],
    "error": ["error"],
    "all_mean_power": [
        "delta_mean_power",
        "theta_mean_power",
        "alpha_mean_power",
        "beta_mean_power",
        "gamma_mean_power",
    ],
    "all_rel_mean_power": [
        "rel_delta_mean_power",
        "rel_theta_mean_power",
        "rel_alpha_mean_power",
        "rel_beta_mean_power",
        "rel_gamma_mean_power",
    ],
    "all_flat_mean_power": [
        "delta_flat_mean_power",
        "theta_flat_mean_power",
        "alpha_flat_mean_power",
        "beta_flat_mean_power",
        "gamma_flat_mean_power",
    ],
    "all_aperiodic": ["aperiodic_offset", "aperiodic_exponent"],
    "all_feats": [
        "delta_mean_power",
        "theta_mean_power",
        "alpha_mean_power",
        "beta_mean_power",
        "gamma_mean_power",
        "rel_delta_mean_power",
        "rel_theta_mean_power",
        "rel_alpha_mean_power",
        "rel_beta_mean_power",
        "rel_gamma_mean_power",
        "delta_flat_mean_power",
        "theta_flat_mean_power",
        "alpha_flat_mean_power",
        "beta_flat_mean_power",
        "gamma_flat_mean_power",
        "aperiodic_offset",
        "aperiodic_exponent",
    ],
}

n_splits = 50
cvs = {
    "StratifiedKFold": StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=42
    ),
    "LeaveOneGroupOut": LeaveOneGroupOut(),
    "Leave2Out": GroupShuffleSplit(
        n_splits=n_splits, test_size=2, random_state=42
    ),  # Leave 2 subjs out
    "LeavePartialGroupOut": LeavePartialGroupOut(
        n_groups=4, p_split=0.5, n_iterations=n_splits, random_state=42
    ),  # Leave 50% of 4 subjs out
}

classifiers = {
    "svm": (
        "svm",
        SVC(kernel="rbf", C=0.1, gamma="scale", class_weight="balanced"),
    ),
    "xgboost": (
        "xgb",
        XGBClassifier(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.05,
            subsample=0.7,
            colsample_bytree=1.0,
            gamma=1.0,
            reg_lambda=10.0,
            reg_alpha=5.0,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
    ),
}


all_results = []
for clf_name, clf in classifiers.items():
    print(f"Classifier: {clf_name}")
    for cv_name, cv in cvs.items():
        print(f"Evaluating classifiers with {cv_name} cross-validation:")
        result_df = compare_features(
            dataset=dataset,
            classifier=clf,
            features=features,
            cv=cv,
            normalize_data=False,
            shuffle_labels=True,
        )
        result_df["classifier_key"] = clf_name
        result_df["classifier_label"] = clf_name
        result_df["cv"] = cv_name
        all_results.append(result_df)

# save results
final_shuffled_results_df = pd.concat(all_results, ignore_index=True)
final_shuffled_results_df.to_csv(
    f"{MODEL_RESULTS_PATH}/model_comparison_shuffled_results.tsv", sep="\t", index=False
)
print(
    "Model comparison complete. Results saved to model_comparison_shuffled_results.tsv"
)